# Notebook 1 — Resume Role Classifier (TF-IDF baseline vs fine-tuned DistilBERT)

**Project:** GenAI Resume Analyser
**Input:** a full resume text → **Output:** the job role (e.g. "Data Science", "Java Developer")

## Syllabus topics this notebook covers
1. **NLP preprocessing** — regex cleaning, lowercasing, whitespace handling, stop words
2. **Classical ML text classification** — TF-IDF vectorization + Logistic Regression
3. **Transformers & fine-tuning** — DistilBERT fine-tuned for sequence classification
4. **Evaluation methodology** — accuracy vs macro-F1, stratified splits, confusion matrices

## Dataset: snehaanbhawal "Resume Dataset"
2,400+ resumes from livecareer.com, **24 balanced role categories** (100 per category),
plain-text `Resume_str` + `Category` in `data.json`. PDF-extracted text is a bit noisy —
the cleaning cell handles that, and it makes the preprocessing step real rather than cosmetic.

## How to run
1. Colab menu: `Runtime` → `Change runtime type` → **T4 GPU** → Save
2. `Runtime` → `Run all` (the dataset cell needs Kaggle secrets OR a manual upload — see that cell)
3. Total runtime: **~25–40 min** (training is ~15–25 min on T4)
4. The last cell downloads `role_classifier_artifacts.zip` — save it into the project's `notebooks/artifacts/` folder

## What good results look like
- TF-IDF + Logistic Regression: **accuracy 0.90–0.98, macro-F1 0.88+**
- Fine-tuned DistilBERT: **accuracy 0.90–0.98, macro-F1 0.88+** (the gap between the two models is usually small on this keyword-rich dataset — that is a legitimate finding, see the examiner section)
- If **any** macro-F1 is below 0.80 → something broke: check that the split is stratified and that `Category` had no unexpected values in the load cell.

In [ ]:
# Install the libraries we need; torch is NOT pinned so Colab's preinstalled CUDA-compatible build is kept
!pip -q install "transformers>=4.44,<5" "datasets>=2.20,<4" "accelerate>=0.34" "scikit-learn>=1.4" "kagglehub" matplotlib seaborn

In [ ]:
# Import everything and fix random seeds so re-runs give identical numbers
import os, re, json, glob, shutil, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
SEED = 42  # one seed everywhere: numpy, python, and the transformers Trainer all use it
random.seed(SEED); np.random.seed(SEED)

## 1. Get the dataset
**Kaggle way (recommended):** add your Kaggle username + API key to Colab secrets.
1. On Kaggle: Settings → API → Create New Token → open `kaggle.json`, it contains your username and key
2. In Colab: click the 🔑 icon in the left sidebar → add `KAGGLE_USERNAME` and `KAGGLE_KEY`

**Manual way (no credentials needed):** download [snehaanbhawal's Resume Dataset](https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset)
— you only need its `data.json` file — then run this cell; it opens an upload button when the Kaggle download fails.

In [ ]:
# Download the dataset via kagglehub; if credentials are missing it falls back to a manual upload dialog
try:
    os.environ["KAGGLEHUB_CACHE"] = "/content/kagglehub"
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab secrets")
    import kagglehub
    base = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
    cands = glob.glob(os.path.join(base, "**", "*.json"), recursive=True) + glob.glob(os.path.join(base, "**", "*.csv"), recursive=True)
    assert cands, "No JSON/CSV found inside the downloaded dataset"
    named = [c for c in cands if os.path.basename(c).lower() == "data.json"]
    DATA_PATH = (named or cands)[0]
except Exception as e:
    print(f"Kaggle download unavailable ({e}) -> manual upload mode")
    from google.colab import files
    up = files.upload()
    DATA_PATH = list(up.keys())[0]
print("Using dataset file:", DATA_PATH)

In [ ]:
# Load the dataset: it ships as data.json (a list of records with Resume_str + Category); a CSV upload also works
if str(DATA_PATH).lower().endswith(".json"):
    try:
        rows = json.load(open(DATA_PATH, encoding="utf-8"))                    # normal case: one big JSON array
    except json.JSONDecodeError:
        rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]  # fallback: JSON-lines format
    df = pd.DataFrame(rows)
else:
    df = pd.read_csv(DATA_PATH)

# Find the text and category columns whatever their casing is (Resume_str / resume_str / Resume / Category...)
lower_map = {c.strip().lower(): c for c in df.columns}
text_col = lower_map.get("resume_str") or lower_map.get("resume") or lower_map.get("text")
cat_col = lower_map.get("category") or lower_map.get("label")
assert text_col and cat_col, f"Could not find resume/category columns, got: {list(df.columns)}"
df = df.rename(columns={text_col: "Resume", cat_col: "Category"})
df = df[df["Resume"].notna()]
df["Category"] = df["Category"].astype(str).str.strip().str.title()            # dataset stores UPPERCASE labels; prettify once here
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nClass distribution:\n", df["Category"].value_counts())
print("\nNulls:", df[["Resume", "Category"]].isnull().sum().to_dict())
print("\nSample text (first 400 chars):\n", str(df["Resume"].iloc[0])[:400])

In [ ]:
# Explore: bar chart of the class distribution, saved as a PNG for the report page
plt.figure(figsize=(10, 6))
df["Category"].value_counts().plot(kind="barh", color="#4c72b0")
plt.title("Resume count per role category")
plt.xlabel("Number of resumes")
plt.tight_layout()
os.makedirs("/content/artifacts/role_classifier", exist_ok=True)
plt.savefig("/content/artifacts/role_classifier/class_distribution.png", dpi=150)
plt.show()
print(f"Unique categories: {df['Category'].nunique()} | total rows: {len(df)}")

## 2. Preprocessing
We strip URLs, emails, non-letter characters and extra whitespace. Both models get the **same** cleaned text so the comparison stays fair. (BERT could eat raw text, but identical inputs keep the experiment clean.)

In [ ]:
# Clean every resume: drop URLs/emails/symbols, collapse whitespace, lowercase
def clean_text(t):
    t = str(t)
    t = re.sub(r"http\S+|www\.\S+", " ", t)   # URLs
    t = re.sub(r"\S+@\S+", " ", t)            # email addresses
    t = re.sub(r"[^A-Za-z ]+", " ", t)        # everything that is not a letter
    return re.sub(r"\s+", " ", t).strip().lower()

df["clean"] = df["Resume"].apply(clean_text)
df = df[df["clean"].str.len() > 50].reset_index(drop=True)  # drop degenerate rows whose text vanished in cleaning
print("Rows after cleaning:", len(df))
print("Before ->", df["Resume"].iloc[0][:120])
print("After  ->", df["clean"].iloc[0][:120])

In [ ]:
# Stratified 80/20 split: keeps the same class proportions in train and test, which matters with 24 classes
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["Category"], random_state=SEED)
print(f"Train: {len(train_df)}  Test: {len(test_df)}")
print("Train classes:", train_df["Category"].nunique(), "| Test classes:", test_df["Category"].nunique())

## 3. Baseline: TF-IDF + Logistic Regression
TF-IDF turns each resume into a weighted bag of words (rare-but-informative words get high weight); Logistic Regression learns one weight vector per role. This is the **classical ML** syllabus topic.

In [ ]:
# TF-IDF (top 5000 word+bigram features, English stop words removed) -> Logistic Regression, trained on the clean text
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])
baseline.fit(train_df["clean"], train_df["Category"])
baseline_pred = baseline.predict(test_df["clean"])
baseline_acc = accuracy_score(test_df["Category"], baseline_pred)
baseline_f1 = f1_score(test_df["Category"], baseline_pred, average="macro")
print(f"Baseline  accuracy: {baseline_acc:.4f}  macro-F1: {baseline_f1:.4f}")

In [ ]:
# Per-class report + confusion matrix heatmap for the baseline
print(classification_report(test_df["Category"], baseline_pred))
cm = confusion_matrix(test_df["Category"], baseline_pred, labels=sorted(df["Category"].unique()))
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=sorted(df["Category"].unique()), yticklabels=sorted(df["Category"].unique()))
plt.title("Confusion matrix — TF-IDF + Logistic Regression")
plt.ylabel("True role"); plt.xlabel("Predicted role"); plt.tight_layout()
plt.savefig("/content/artifacts/role_classifier/cm_baseline.png", dpi=150)
plt.show()

## 4. Fine-tuned DistilBERT
DistilBERT is a 6-layer transformer (a smaller, faster BERT). Fine-tuning means we add one classification head and update the pretrained weights on our 962 resumes. This is the **transformers & fine-tuning** syllabus topic.

In [ ]:
# Convert the pandas frames into Hugging Face Datasets with integer labels, then tokenize (max 256 tokens covers most resumes)
from datasets import Dataset
from transformers import AutoTokenizer
labels = sorted(df["Category"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
ds_train = Dataset.from_pandas(train_df[["clean", "Category"]].rename(columns={"clean": "text", "Category": "label_str"}))
ds_test = Dataset.from_pandas(test_df[["clean", "Category"]].rename(columns={"clean": "text", "Category": "label_str"}))
def add_ids(ex): return {"labels": label2id[ex["label_str"]]}
ds_train = ds_train.map(add_ids); ds_test = ds_test.map(add_ids)
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(batch): return tok(batch["text"], truncation=True, max_length=256)
ds_train = ds_train.map(tokenize, batched=True)
ds_test = ds_test.map(tokenize, batched=True)

In [ ]:
# Build the model (random classification head on pretrained DistilBERT) and the Trainer with per-epoch checkpoints
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import transformers
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(labels), id2label=id2label, label2id=label2id
)
def compute_metrics(eval_pred):
    logits, y = eval_pred
    p = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(y, p), "f1_macro": f1_score(y, p, average="macro")}
args = TrainingArguments(
    output_dir="/content/checkpoints/role_classifier",  # checkpoints land here every epoch, so a Colab timeout can be resumed
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, num_train_epochs=4,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    weight_decay=0.01, warmup_ratio=0.1, fp16=True,
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    save_total_limit=2, logging_steps=20, report_to="none", seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_test,
                  processing_class=tok, compute_metrics=compute_metrics)
print("transformers", transformers.__version__, "| model ready")

In [ ]:
# Train — automatically resumes from the newest checkpoint if the runtime restarted mid-training
ckpt_dirs = [os.path.join(args.output_dir, d) for d in os.listdir(args.output_dir) if d.startswith("checkpoint-")] if os.path.isdir(args.output_dir) else []
last = max(ckpt_dirs, key=os.path.getmtime) if ckpt_dirs else None
trainer.train(resume_from_checkpoint=last)

In [ ]:
# Evaluate the best checkpoint on the held-out test set and inspect a few example predictions
bert_pred_ids = trainer.predict(ds_test).predictions.argmax(axis=-1)
bert_pred = [id2label[i] for i in bert_pred_ids]
bert_acc = accuracy_score(test_df["Category"], bert_pred)
bert_f1 = f1_score(test_df["Category"], bert_pred, average="macro")
print(f"DistilBERT  accuracy: {bert_acc:.4f}  macro-F1: {bert_f1:.4f}")
for txt, true_l, pred_l in list(zip(test_df["clean"], test_df["Category"], bert_pred))[:5]:
    print(f"  true={true_l:22s} pred={pred_l:22s} {'OK' if true_l == pred_l else 'WRONG'}")

In [ ]:
# Build the comparison table (accuracy + macro-F1 for both models) and save it as CSV
comparison = pd.DataFrame([
    {"model": "TF-IDF + Logistic Regression", "accuracy": round(baseline_acc, 4), "macro_f1": round(baseline_f1, 4)},
    {"model": "Fine-tuned DistilBERT", "accuracy": round(bert_acc, 4), "macro_f1": round(bert_f1, 4)},
])
comparison.to_csv("/content/artifacts/role_classifier/model_comparison.csv", index=False)
print(comparison.to_string(index=False))

In [ ]:
# Confusion matrix for DistilBERT, saved as a PNG for the Model Performance page
cm2 = confusion_matrix(test_df["Category"], bert_pred, labels=labels)
plt.figure(figsize=(11, 9))
sns.heatmap(cm2, annot=True, fmt="d", cmap="Greens", xticklabels=labels, yticklabels=labels)
plt.title("Confusion matrix — fine-tuned DistilBERT")
plt.ylabel("True role"); plt.xlabel("Predicted role"); plt.tight_layout()
plt.savefig("/content/artifacts/role_classifier/cm_distilbert.png", dpi=150)
plt.show()

In [ ]:
# Save model + tokenizer + label map + metrics as JSON, then zip everything into one downloadable artifact
out = "/content/artifacts/role_classifier"
trainer.save_model(out + "/model")
tok.save_pretrained(out + "/model")
with open(out + "/labels.json", "w") as f:
    json.dump({"labels": labels, "label2id": label2id, "id2label": {str(k): v for k, v in id2label.items()}}, f, indent=2)
with open(out + "/metrics.json", "w") as f:
    json.dump({
        "task": "role_classification",
        "n_train": len(train_df), "n_test": len(test_df), "n_classes": len(labels),
        "baseline": {"model": "tfidf_logreg", "accuracy": round(baseline_acc, 4), "macro_f1": round(baseline_f1, 4)},
        "distilbert": {"model": "distilbert-base-uncased-finetuned", "accuracy": round(bert_acc, 4), "macro_f1": round(bert_f1, 4)},
    }, f, indent=2)
shutil.make_archive("/content/role_classifier_artifacts", "zip", out)
print("Zipped:", os.path.getsize("/content/role_classifier_artifacts.zip") // (1024 * 1024), "MB")

In [ ]:
# Trigger the browser download of the artifact zip — move it into D:\resume-analyser\notebooks\artifacts\
from google.colab import files
files.download("/content/role_classifier_artifacts.zip")

## Explain this to my examiner

**What the model is.** Two classifiers trained on the same cleaned resumes (2,400+ resumes, 24 balanced categories): (1) a classical pipeline where TF-IDF converts text into 5000 weighted word/bigram features and Logistic Regression learns linear decision boundaries per role; (2) DistilBERT — a pretrained 6-layer transformer — fine-tuned end-to-end with a new classification head, using a 2e-5 learning rate, batch size 16, 4 epochs, and half-precision (fp16) math on the free T4 GPU.

**What the metrics mean.** *Accuracy* is the fraction of resumes assigned the correct role, but it hides weak classes; *macro-F1* is the unweighted average of per-class F1 (harmonic mean of precision and recall), so a model that ignores a small class gets punished. I used a **stratified split** so every class appears in train and test in the same proportion, and the **confusion matrix** shows exactly which role pairs the model confuses.

**Syllabus topics covered.** NLP preprocessing (regex cleaning, lowercasing, stop-word removal), classical ML text classification (TF-IDF + Logistic Regression), transformers and fine-tuning (DistilBERT on a downstream task), embeddings (TF-IDF sparse embeddings vs contextual BERT embeddings), and evaluation methodology (accuracy vs macro-F1, stratified sampling, confusion matrices).

**One honest finding to say out loud.** On this keyword-rich dataset, TF-IDF + Logistic Regression often matches or slightly beats DistilBERT — role-specific keywords (e.g. "SAP", "React") are near-perfect linear discriminators, and the PDF-extracted text noise limits everyone equally. Fine-tuning wins when training data is larger still or the text is more ambiguous. Reporting that openly is better ML than claiming "transformer = always better".

## Troubleshooting
- **`Kaggle download unavailable`** → the manual upload dialog opened; pick `data.json` from the dataset you downloaded (see the link in the "Get the dataset" cell).
- **CUDA out of memory** → set `per_device_train_batch_size=8` in the TrainingArguments cell and rerun from there.
- **Runtime restarted / session died** → `Runtime → Run all` again; the training cell auto-resumes from the last epoch checkpoint. (Checkpoints live in `/content`, which is wiped on a full session reset — for long sessions mount Drive and change `output_dir` to a Drive path.)
- **Wrong zip downloaded** → re-run the save cell; the zip is always rebuilt fresh from `/content/artifacts/role_classifier`.